# Memory

In [ ]:
# lanchain.memory
# 대부분 0.3.1 부터 deprecated 됨. LangGraph 로의 사용을 권하고 있슴.


# 환경변수

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.prompts.chat import ChatPromptTemplate
from langchain_core.messages.human import HumanMessage
from langchain_core.messages.ai import AIMessage

# 메모리 없는 경우

In [3]:
llm = ChatOpenAI(temperature=0.1)

prompt = ChatPromptTemplate.from_messages([
    ('system', "You are a helpful AI talking to a human"),
    ('human', '{question}'),
])

chain = prompt | llm

def invoke_chain(question):
    result = chain.invoke({'question': question})
    print('🐹', result.content)    

invoke_chain('My name is John')
invoke_chain('What is my name?')


🐹 Hello John! How can I assist you today?
🐹 I'm sorry, but I don't have access to personal information like your name. How can I assist you today?


In [ ]:
# AI 는 요청한 내용에 대한 상태정보를 기억하지 않는다.

# 챗봇은 '대화의 상태'를 기억해야 한다. -> '문맥에 맞는 대화'를 하려면!

# 이전 대화의 내용을 기억하여 AI에 요청해야 한다.

# LangChain 에선 이를 Memory 객체들 제공하여 지원했었다  ~ v0.3.x
#       v1.x.x 부터는 대부분의 Memory 객체들 삭제.  -> LangGraph 를 사용하여 상태정보 유지.

# 메모리 구현 - MessagesPlaceHolder

In [5]:
from langchain_core.prompts.chat import MessagesPlaceholder

llm = ChatOpenAI(temperature=0.1)

chat_history = []   # 챗봇과 나눈 Message 들을 담을 것이다.

prompt = ChatPromptTemplate.from_messages([
    ('system', "You are a helpful AI talking to a human"),

    # 이전의 chat_history 내역이 여기에 들어오게 된다.
    # 'history' 라는 키값으로 전달해주면 된다.
    MessagesPlaceholder(variable_name='history'),
    
    ('human', '{question}'),
])

chain = prompt | llm

def invoke_chain(question, history):
    result = chain.invoke({'question': question, "history": chat_history})
    print('🐹', result.content)    
    # 요청-응답 내역을 history 에 추가해야 한다.
    history.append(HumanMessage(content=question))
    history.append(AIMessage(content=result.content))
    

print('🔷 chat_history:', chat_history)
invoke_chain("My name is Harang", chat_history)
print('🔷 chat_history:', chat_history)
invoke_chain("What is my name?", chat_history)
print('🔷 chat_history:', chat_history)


🔷 chat_history: []
🐹 Hello Harang! How can I assist you today?
🔷 chat_history: [HumanMessage(content='My name is Harang', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello Harang! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
🐹 Your name is Harang.
🔷 chat_history: [HumanMessage(content='My name is Harang', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello Harang! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}), AIMessage(content='Your name is Harang.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [ ]:
"""
잘 동작하긴 하나 위 코드의 경우
매번 chain 을 호출할때 chat_history 를 넘겨주어야 한다.
  invoke_chain("My name is Harang", chat_history) <-- !!

자동으로 chat history  를 건네주려면?  ↓↓↓
"""
None

# 메모리 구현 - RunnablePassthrough

In [8]:
from langchain_core.prompts.chat import MessagesPlaceholder
from langchain_core.runnables.passthrough import RunnablePassthrough

llm = ChatOpenAI(temperature=0.1)

chat_history = []   # 챗봇과 나눈 Message 들을 담을 것이다.

prompt = ChatPromptTemplate.from_messages([
    ('system', "You are a helpful AI talking to a human"),

    # 이전의 chat_history 내역이 여기에 들어오게 된다.
    # 'history' 라는 키값으로 전달해주면 된다.
    MessagesPlaceholder(variable_name='history'),
    
    ('human', '{question}'),
])

# ✅ chain 호출할때마다 과거 대화내역이 자동으로 llm 에 전달되게 하려면!

def load_memory(_):
    # print('🎃', xxx)
    return chat_history

chain = RunnablePassthrough.assign(history=load_memory) | prompt | llm

def invoke_chain(question, history):
    result = chain.invoke({'question': question})  # ✅ invoke 호출코드에서 chat_history 를 매번 입력하는게 불편했다 
    print('🐹', result.content)    
    # 요청-응답 내역을 history 에 추가해야 한다.
    history.append(HumanMessage(content=question))
    history.append(AIMessage(content=result.content))
    

print('🔷 chat_history:', chat_history)
invoke_chain("My name is Harang", chat_history)
print('🔷 chat_history:', chat_history)
invoke_chain("What is my name?", chat_history)
print('🔷 chat_history:', chat_history)


🔷 chat_history: []
🐹 Hello Harang! How can I assist you today?
🔷 chat_history: [HumanMessage(content='My name is Harang', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello Harang! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
🐹 Your name is Harang.
🔷 chat_history: [HumanMessage(content='My name is Harang', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello Harang! How can I assist you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='What is my name?', additional_kwargs={}, response_metadata={}), AIMessage(content='Your name is Harang.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
